# Asari & Zador (2009), Fig. 4: STRF and LN model performance on Wehr's A1 cells

Paper: H. Asari and A. M. Zador (2009), "Long-Lasting Context Dependence Constrains Neural
Encoding Models in Rodent Auditory Cortex", J Neurophysiol 102:2638-2656, doi:10.1152/jn.00577.2009.

Data set: CRCNS ac-1 (unversioned; doi:10.6080/K0KW5CXR), Wehr's recordings (`wehr/` in
`crcns-ac1.zip`), read in place from the public S3 bucket `crcnsarchive` over HTTP range
requests; nothing is downloaded to disk.

This notebook tests one claim, `fig4-strf-performance-low`: linear STRF models (and LN models with
a static nonlinearity) of A1 cells from Machens et al. (2004) predict less than about 20% of the
predictable response power at every window length from 180 ms to about 4 s, and extending the window
does not improve them significantly. Methods follow the paper (Methods, "Neural encoding models",
Eqs. 46 to 51; "Model performance", Eqs. 52 and 53), the claim's criterion, and the authors' own code
shipped in the zip (`wehr/Tools/STRFanalysis.m`, `ridgesvd1.m`, `logspectrogram.m`,
`ViewResponse.m`, `GetEvoked.m`, `MakeRamp.m`). Where each parameter comes from is listed in the
docstring of `wehr.py`; the assumptions are repeated below.

In [ ]:
import os, time, zlib, resource
os.environ.setdefault("OMP_NUM_THREADS", "1")       # two worker processes, one BLAS thread each
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import wehr as W

T0 = time.time()
CLAIM = "asari-2009/fig4-strf-performance-low"
SEED = zlib.crc32(CLAIM.encode())   # defined by the protocol; no result here uses random draws
print("seed", SEED, "(unused: folds are contiguous blocks or stimuli in order)")
np.set_printoptions(precision=3, suppress=True, linewidth=140)

## Data

`wehr/Summary.mat` lists two experiments, both citing Machens et al. (2004): ten cells (2002) with a
fixed subset of natural sounds repeated 5 to 25 times, and sixteen cells (2003) with many natural
sound streams presented once or twice. The paper's 20 cells are not marked in the files. Eq. 52
needs at least two trials of a stimulus, so we keep every cell with at least two trials of some
natural sound, and only those stimuli (an assumption). Each recording file is one trial; the trace
is median filtered (10 ms, `ViewResponse.m`/`GetEvoked.m`), cut to the stimulus duration, averaged
in 4-ms bins and mean-subtracted per trial (`STRFanalysis.m`). Spectrograms of every stimulus are
computed with both window readings (10-ms and 20-ms Hann).

In [ ]:
t = time.time()
rows = W.summary()
allcells = sorted({r["cell"] for r in rows})
SEL = W.select_cells(min_trials=2)
print("Summary.mat: experiments", sorted({(r["experiment"], r["exp_description"]) for r in rows}))
print("references:", sorted({r["reference"] for r in rows}))
print("cells in Summary.mat:", len(allcells), "| cells with >= 2 trials of a natural sound:", len(SEL))
print("excluded (every natural sound presented once):",
      [c for c in allcells if c not in SEL])
CELLS, STIM = W.load_cells(SEL, wins_ms=(10, 20), threads=4)
print(f"loaded in {time.time() - t:.0f} s; peak RSS {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
print()
print(f"{'cell':16s} {'stim':>4s} {'trials per stimulus':40s} {'bins':>7s} {'sf':>6s} unit  raw range (mV)   stim dur (s)")
for c, st in CELLS.items():
    tr = [s["R"].shape[0] for s in st]
    nb = sum(s["R"].shape[1] for s in st)
    lo = min(s["raw_range"][0] for s in st); hi = max(s["raw_range"][1] for s in st)
    durs = sorted({round(s["duration"], 2) for s in st})
    print(f"{c:16s} {len(st):4d} {str(tr):40.40s} {nb:7d} {st[0]['sf']:6.0f} {st[0]['unit']:4s} "
          f"[{lo:6.1f}, {hi:5.1f}]  {durs}")
print()
print("stimuli:", len(STIM))
for f, s in sorted(STIM.items()):
    print(f"  {f:22s} sf {s['sf']:.0f} Hz, {s['dur']:.2f} s, peak |x| {s['peak']:.2f}, "
          f"spectrogram (10 ms) {s[10].shape}, dB range after centring "
          f"[{s[10].min():.0f}, {s[10].max():.0f}]")
print("frequencies (Hz):", np.round(W.FREQS).astype(int))
print("windows (ms):", W.WINDOWS_MS)
print("bin splits:", W.BIN_SPLITS)

## Claim `fig4-strf-performance-low`

> Linear STRF models of 20 A1 cells from Machens et al. (2004) predicted less than about 20% of the
> predictable response power, and extending the window from 180 ms to about 4 s did not improve them
> significantly, with or without a static nonlinearity; the LN model did only slightly better.

Expected: STRF model performance at any window length, upper bound 0.2, tolerance 0.02.
Reproduced when, under one reading, the mean performance is at most 0.22 at every window for STRF and
LN, and a paired one-sided Wilcoxon signed-rank test (validation performance at 4052 ms against
180 ms, over cells, at 0.05) finds no improvement for either; a Friedman test over the nine windows
is a variant that does not decide.

Method (source in brackets): spectrogram of Eq. 47 at 17 frequencies 400 * 2^(k/3) Hz, k = 0..16
[paper: 3 bins/octave from 0.40 kHz, 17 bins; `logspectrogram.m` evaluates the DFT at each centre
frequency; assumption: its rule would give 18 bins up to 20.3 kHz, we keep 17], centred per frequency
in each stimulus [`STRFanalysis.m`]; 45 history bins of 2^k ms split as in the Methods [paper];
4-ms response bins, spectrogram frames every 4 ms with the window centred on the bin, coarse bins the
mean of the 4-ms frames they cover [assumption]; history before onset set to the stimulus mean
in the "all" readings [assumption; every window is scored on the same bins]; ridge regression (Eq. 50) on the
trial-averaged response with 33 ridge values 10^-6 ... 10^2 times the mean eigenvalue of the training
S'S [assumption: grid]; 10 folds [paper], one ridge value for all folds chosen by the mean validation
performance of the linear model, and the LN model uses that filter [`STRFanalysis.m`]; robust lowess
(5% span, one fit plus five robustness iterations) on the training predictions, applied to the
validation fold [paper; iteration convention an assumption]; Eq. 53 with Eq. 52 per fold as in
`LNperformance` [code], averaged over folds and then over cells [assumption / paper Fig. 4 legend].

Readings run (6 in all, each read on validation and on training data): spectrogram window 10-ms or
20-ms Hann (criterion), crossed with three fold schemes: 10 contiguous blocks of the concatenated
stimuli (`STRFanalysis.m`) scoring every bin from stimulus onset with the pre-onset history set to the
stimulus mean ("contig/all"); folds by stimulus (stimulus i in fold i mod 10, fewer folds if a cell
has fewer than 10 stimuli), all bins ("stim/all"); and contiguous blocks over only the bins at least
4052 ms after onset, so that no window reaches before onset ("contig/late"; `STRFanalysis.m` also
drops the first window length of each stimulus, there for each window separately; here the same bins
for every window so windows stay comparable). The 20% is read
as the validation (lower) or training (upper) estimate. Response sampling: only 4-ms bins were run (Wehr's traces are sampled at 4 kHz, not 10 kHz).

In [ ]:
import multiprocessing as mp
SCHEMES = ["contig/all", "stim/all", "contig/late"]
READINGS = [f"hann{w}/{s}" for w in (10, 20) for s in SCHEMES]
cellnames = list(CELLS)
W._DATA.update(cells=CELLS, stim=STIM)          # inherited by the two forked workers
size = {c: sum(s["R"].shape[1] for s in CELLS[c]) for c in cellnames}
# largest cells first within each spectrogram window, so the two largest fits never run together
tasks = sorted([(c, w) for c in cellnames for w in (10, 20)], key=lambda t: (t[1], -size[t[0]]))
t = time.time()
RES = {r: {} for r in READINGS}
INFO = {}
with mp.get_context("fork").Pool(2) as pool:
    for c, wm, res, snr, nlate, nall in pool.imap_unordered(W.run_cell, tasks):
        for s in SCHEMES:
            RES[f"hann{wm}/{s}"][c] = [res[w][s] for w in range(9)]
        INFO[c] = dict(snr=snr, late=nlate, all=nall)
print(f"fits: {time.time() - t:.0f} s (2 processes, 1 BLAS thread each); "
      f"largest worker peak RSS {resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6:.2f} GB")

def arr(name, model, col):
    # (cells, windows): col 0 = training (upper), 1 = validation (lower); mean over folds
    return np.array([[RES[name][c][w][model][:, col].mean() for w in range(9)] for c in cellnames])

In [ ]:
print("cells:", len(cellnames))
print("folds per cell (by-stimulus readings):", {c: RES["hann10/stim/all"][c][0]["nfold"] for c in cellnames})
print("bins per cell, all / late (>= 4052 ms after onset):", {c: (INFO[c]["all"], INFO[c]["late"]) for c in cellnames})
print("predictable power (Eq. 52) / power of the trial mean, per cell :",
      {c: round(INFO[c]["snr"], 3) for c in cellnames})
print()
print("Mean over cells of model performance (Eq. 53)")
print(f"{'window ms':>10s}", "".join(f"{w:>7d}" for w in W.WINDOWS_MS), "   max")
for name in READINGS:
    print(name)
    for model in (("strf", "ln", "ln_nonrobust") if name.endswith("contig/all") else ("strf", "ln")):
        for col, lab in ((1, "valid"), (0, "train")):
            m = arr(name, model, col).mean(0)
            print(f"{model + ' ' + lab:>18s}", "".join(f"{v:7.3f}" for v in m), f"  {m.max():.3f}")
print()
print("Median over cells (robustness check, does not decide)")
for name in READINGS:
    print(name)
    for model in ("strf", "ln"):
        for col, lab in ((1, "valid"), (0, "train")):
            m = np.median(arr(name, model, col), 0)
            print(f"{model + ' ' + lab:>18s}", "".join(f"{v:7.3f}" for v in m), f"  {m.max():.3f}")

In [ ]:
print("Per-cell validation (lower) and training (upper) performance, reading hann10/contig/all")
for model in ("strf", "ln"):
    for col, lab in ((1, "valid"), (0, "train")):
        a = arr("hann10/contig/all", model, col)
        print(f"{model} {lab}  (rows: cells; columns: windows {W.WINDOWS_MS})")
        for c, r in zip(cellnames, a):
            print(f"  {c:16s}", r)
print()
for name in READINGS[1:]:
    print(f"Per-cell validation at 180 / 4052 ms, reading {name}")
    for model in ("strf", "ln"):
        a = arr(name, model, 1)
        print(" ", model, [f"{c[2:8]}-{c[-3:]}: {x:.3f}/{y:.3f}" for c, x, y in zip(cellnames, a[:, 0], a[:, 8])])

In [ ]:
c = "20020602-mw-001"
Sc, rbar, ppred, sid, rws = W.prepare(CELLS[c], STIM, 10)
late = np.concatenate([np.arange(s["R"].shape[1]) >= W.PAD for s in CELLS[c]])
fl = np.full(len(rws), -1); fl[late] = W.folds_contiguous(int(late.sum()))
print(f"{c}, reading hann10/contig/late, 4052 ms: per validation fold")
r = RES["hann10/contig/late"][c][8]
for f in range(10):
    v = fl == f
    print(f"  fold {f}: predictable power (sum of Eq. 52 over bins) {ppred[v].sum():9.1f}, "
          f"power of trial mean {np.sum(rbar[v] ** 2):9.1f}, STRF validation {r['strf'][f, 1]:8.3f}")
del Sc

In [ ]:
print("Ridge value chosen (index into a 33-point grid, 0..32), per cell and window")
for name in READINGS:
    B = np.array([[RES[name][c][w]["best"] for w in range(9)] for c in cellnames])
    edge = int(((B == 0) | (B == len(W.RIDGE_REL) - 1)).sum())
    print(f"{name}: grid edge hits {edge} of {B.size}; index range {B.min()}..{B.max()}; "
          f"relative ridge 10^{np.log10(W.RIDGE_REL[B.min()]):.2f} .. 10^{np.log10(W.RIDGE_REL[B.max()]):.2f}")
    if name == "hann10/contig/all":
        for c, b in zip(cellnames, B):
            print(f"  {c:16s}", b)
bad = [(n, c, w) for n in READINGS for c in cellnames for w in range(9)
       if not np.all(np.isfinite(RES[n][c][w]["strf"])) or not np.all(np.isfinite(RES[n][c][w]["ln"]))]
print("fits with non-finite performance:", bad, "| cells dropped: none")

In [ ]:
print("Paired Wilcoxon signed-rank, validation performance at 4052 ms vs 180 ms, one-sided (improvement)")
print("and Friedman test over the nine windows (variant, does not decide)")
TESTS = {}
for name in READINGS:
    for model in ("strf", "ln"):
        a = arr(name, model, 1)
        w = stats.wilcoxon(a[:, 8], a[:, 0], alternative="greater")
        w2 = stats.wilcoxon(a[:, 8], a[:, 0])
        fr = stats.friedmanchisquare(*a.T)
        TESTS[(name, model)] = w.pvalue
        print(f"{name:18s} {model:4s}: median diff {np.median(a[:, 8] - a[:, 0]):+.4f}, "
              f"cells improved {int((a[:, 8] > a[:, 0]).sum())}/{len(a)}, W={w.statistic:.0f}, "
              f"p(one-sided)={w.pvalue:.4f}, p(two-sided)={w2.pvalue:.4f}; "
              f"Friedman chi2={fr.statistic:.1f}, p={fr.pvalue:.4f}")
print()
print("LN gain (mean LN minus mean STRF), validation and training, per window")
for name in READINGS:
    gv = arr(name, "ln", 1).mean(0) - arr(name, "strf", 1).mean(0)
    gt = arr(name, "ln", 0).mean(0) - arr(name, "strf", 0).mean(0)
    print(f"{name:18s} valid", gv, " train", gt)
print()
print("Criterion check per reading: max over windows of mean performance <= 0.22 for STRF and LN, "
      "and no Wilcoxon improvement (p >= 0.05) for either")
for name in READINGS:
    for col, lab in ((1, "validation"), (0, "training")):
        mx = max(arr(name, "strf", col).mean(0).max(), arr(name, "ln", col).mean(0).max())
        noimp = TESTS[(name, "strf")] >= 0.05 and TESTS[(name, "ln")] >= 0.05
        print(f"  {name:18s} {lab:10s}: max mean {mx:.3f} ({'<=' if mx <= 0.22 else '>'} 0.22); "
              f"no improvement: {noimp} -> {'meets' if (mx <= 0.22 and noimp) else 'fails'}")

In [ ]:
sub = [i for i, c in enumerate(cellnames) if c.startswith("2002")]
print("Variant (does not decide): only the 2002 cells, 5 to 25 trials per stimulus:", len(sub), "cells")
for name in READINGS:
    for model in ("strf", "ln"):
        v = arr(name, model, 1)[sub]; tr = arr(name, model, 0)[sub]
        w = stats.wilcoxon(v[:, 8], v[:, 0], alternative="greater")
        print(f"  {name:18s} {model:4s} valid {v.mean(0)}  train {tr.mean(0)}  p={w.pvalue:.4f}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
x = np.array(W.WINDOWS_MS) / 1000
for ax, name in zip(axs, ["hann10/contig/all", "hann10/contig/late"]):
    for model, col in (("strf", "0.55"), ("ln", "k")):
        ax.plot(x, arr(name, model, 1).mean(0), "x-", color=col, label=f"{model.upper()} validation")
        ax.plot(x, arr(name, model, 0).mean(0), "o--", mfc="none", color=col, label=f"{model.upper()} training")
    ax.axhline(0.2, color="r", lw=0.8, ls=":")
    ax.set_title(name); ax.set_xlabel("window length (s)")
axs[0].set_ylabel("model performance (Eq. 53)"); axs[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Result: not reproduced under the criterion. The validation (lower) estimate is low at every
window in every reading that scores all bins, but in each of those readings the paired Wilcoxon test
finds that the STRF model improves from 180 ms to 4052 ms.**

Cells: 16 Wehr cells with at least two trials of a natural sound (Eq. 52 needs repeats); the paper's
20 cells are not marked in the files, and the 9 cells whose sounds were all presented once are left
out. Under the four readings that score every bin (10-ms or 20-ms Hann window, contiguous folds or
folds by stimulus), the mean validation performance is at most 0.148 for STRF and at most 0.111 for
LN at every window, both below 0.22. Training (upper) estimates reach 0.245 to 0.268 for STRF and
0.281 to 0.311 for LN, above 0.22, so the 20% bound holds only when read as the validation estimate.

The windows do matter. With 10-ms Hann and contiguous folds the mean STRF validation performance goes
from 0.115 at 180 ms to 0.147 at 4052 ms (0.143 by 548 ms, then flat), and the one-sided Wilcoxon
p-value for STRF is 0.0026; it is 0.0011, 0.037 and 0.029 in the other three such readings. For LN,
p is 0.017 and 0.015 with contiguous folds and 0.11 and 0.25 with folds by stimulus. The Friedman
variant (does not decide) is mixed: STRF p 0.016 to 0.31, LN p 0.0011 to 0.40.

Scoring only bins at least 4052 ms after onset ("contig/late") removes the significant change
(STRF p 0.065 and 0.052, LN p 0.14 and 0.12), but there the mean is not a usable summary: one cell,
20020602-mw-001, reaches 14.7 at 4052 ms because the predictable power (Eq. 52) summed over a validation fold is negative in 6 of its 10 folds (printed per fold),
which drives the mean to 1.26 (medians at most 0.195), so that reading fails the 0.22 bound.

The LN model is slightly better than STRF on training data (gain +0.013 to +0.049 across readings
and windows) but worse on validation data (gain -0.025 to -0.34): the robust lowess fitted to
in-sample predictions does not carry over to held-out folds; without the robustness iterations
(diagnostic) the LN validation mean is still at most 0.043 with contiguous folds. The ridge value hit
the grid edge in 9 or 10 of 144 fits in each all-bins reading, all at the strongest ridge (with
10-ms Hann and contiguous folds these are cell 20020608-mw-001 at every window, whose validation
performance stays near zero), and in 24 or 25 fits in the "late" readings, at both ends; no fit
gave a non-finite value and no cell was dropped.

In [ ]:
print(f"total time {time.time() - T0:.0f} s; peak RSS {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB "
      f"(this process), {resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6:.2f} GB (largest worker)")